# 🧬 Expresiones Regulares para Búsqueda de Patrones Biológicos

**Material de Clase | Ciencias de la Computación**
*Sesión en equipo — 30 minutos*

---

**Equipo:** _______________
**Integrantes:** _______________________________________________

In [ ]:
import re

def buscar_y_mostrar(patron, secuencia, descripcion=""):
    if descripcion:
        print(f"📌 {descripcion}")
    print(f"   Secuencia : {secuencia}")
    print(f"   Patrón    : {patron}")
    resultados = list(re.finditer(patron, secuencia))
    if resultados:
        for r in resultados:
            print(f"   Encontrado: '{r.group()}' en posición {r.start()}")
        print(f"   Total     : {len(resultados)} coincidencia(s)")
    else:
        print("   Resultado : Sin coincidencias")
    print()

print("✅ Todo listo")

---
## 🔴 Bloque 1: Búsqueda en Secuencias de DNA

| Señal biológica | Secuencia | Función |
|-----------------|-----------|---------|
| Codón de inicio | `ATG` | Dónde empieza una proteína |
| Codones de paro | `TAA`, `TAG`, `TGA` | Dónde termina una proteína |
| Sitio EcoRI | `GAATTC` | Reconocido por la enzima EcoRI |
| Caja TATA | `TATAAAA` | Región promotora del gen |

In [ ]:
DNA = "TATAAAAATGCCCGAATTCAAATGGTTTAAGCTATGCAGTAGTAATGAAACGATGA"
print(f"Secuencia ({len(DNA)} nt):")
print(DNA)

In [ ]:
# Ejemplo 1.1: Codones de inicio
buscar_y_mostrar("ATG", DNA, "Codones de inicio ATG")

In [ ]:
# Ejemplo 1.2: Codones de paro — alternancia con |
buscar_y_mostrar("TAA|TAG|TGA", DNA, "Codones de paro")

In [ ]:
# Ejemplo 1.3: Sitio de restricción EcoRI
# La enzima corta el DNA en GAATTC
buscar_y_mostrar("GAATTC", DNA, "Sitios de restricción EcoRI")

In [ ]:
# Ejemplo 1.4: ORFs — búsqueda NO codiciosa (+?)
# +? encuentra el ORF más corto posible (hasta el primer codón de paro)
patron_orf = r"ATG[ATCG]+?(TAA|TAG|TGA)"

print("ORFs encontrados:")
for i, match in enumerate(re.finditer(patron_orf, DNA), 1):
    orf = match.group()
    longitud = len(orf)
    print(f"\nORF #{i}: {orf}")
    print(f"  Posición  : {match.start()} — {match.end()}")
    print(f"  Longitud  : {longitud} nt")
    print(f"  Codón paro: {match.group(1)}")
    print(f"  Válido ÷3 : {'✅ Sí' if longitud % 3 == 0 else '⚠️ No'}")

### 💡 Codicioso vs. No Codicioso

| Modo | Símbolo | Comportamiento |
|------|---------|----------------|
| **Codicioso** (greedy) | `+`, `*` | Coincidencia **más larga** posible |
| **No codicioso** (lazy) | `+?`, `*?` | Coincidencia **más corta** posible |

En biología: con `+` obtenemos el ORF más largo (puede saltar codones de paro internos). Con `+?` encontramos el ORF hasta el **primer** codón de paro.

In [ ]:
# Demostración
seq_demo = "ATGAAATAAATGCCC"
print(f"Secuencia: {seq_demo}\n")

m_greedy = re.search(r"ATG[ATCG]+(TAA|TAG|TGA)", seq_demo)
m_lazy   = re.search(r"ATG[ATCG]+?(TAA|TAG|TGA)", seq_demo)

print(f"Codicioso (+)    : '{m_greedy.group() if m_greedy else 'sin coincidencia'}'")
print(f"No codicioso (+?): '{m_lazy.group()   if m_lazy   else 'sin coincidencia'}'")

---
## 🟡 Bloque 2: Motivos en Secuencias de Proteínas

| Motivo | Patrón | Función |
|--------|--------|---------|
| Señal NLS | `[KR]{2,}` | Lleva la proteína al núcleo |
| N-glicosilación | `N[^P][ST]` | Punto de unión de azúcares |
| Dominio TM | `[AVILMFWP]{15,}` | Posible hélice transmembrana |
| Señal KDEL | `[KHR]DEL$` | Retención en retículo endoplásmico |
| Motivo RGD | `RGD` | Adhesión celular |

In [ ]:
PROTEINA = "MKKRGDSTVNASTKDELPVIAVILMFFWPLLSATKNLSNITSIKDEL"
print(f"Proteína ({len(PROTEINA)} aa):")
print(PROTEINA)

In [ ]:
# 2.1 Señal de localización nuclear (NLS)
buscar_y_mostrar("[KR]{2,}", PROTEINA, "Regiones ricas en K/R (posible NLS)")

In [ ]:
# 2.2 Sitio de N-glicosilación — patrón PROSITE: N-{P}-[ST]
# N + cualquier aminoácido excepto P + S o T
buscar_y_mostrar("N[^P][ST]", PROTEINA, "Sitios de N-glicosilación")

In [ ]:
# 2.3 Dominio transmembrana: región hidrofóbica larga
buscar_y_mostrar("[AVILMFWP]{8,}", PROTEINA, "Región hidrofóbica (posible dominio TM)")

In [ ]:
# 2.4 Señal KDEL al final de la secuencia
buscar_y_mostrar("[KHR]DEL$", PROTEINA, "Señal de retención en RE (KDEL/HDEL/RDEL)")

---
## 🟢 Bloque 3: Técnicas Adicionales

In [ ]:
# Grupos con nombre — código más legible
# Sintaxis: (?P<nombre>patrón)
patron_orf_nombrado = r"(?P<inicio>ATG)(?P<cuerpo>[ATCG]+?)(?P<paro>TAA|TAG|TGA)"

print("ORFs con grupos nombrados:")
for i, match in enumerate(re.finditer(patron_orf_nombrado, DNA), 1):
    print(f"\nORF #{i}:")
    print(f"  Inicio : {match.group('inicio')}")
    print(f"  Cuerpo : {match.group('cuerpo')}")
    print(f"  Paro   : {match.group('paro')}")

In [ ]:
# re.IGNORECASE: ignorar mayúsculas/minúsculas
secuencia_mezclada = "ATGcccGAATTcaaATGgtTTAAGCTATG"

sin_flag = re.findall("ATG", secuencia_mezclada)
con_flag = re.findall("ATG", secuencia_mezclada, re.IGNORECASE)
print(f"Sin re.IGNORECASE: {sin_flag}")
print(f"Con re.IGNORECASE: {con_flag}")

In [ ]:
# re.sub: limpiar secuencias con formato sucio
secuencia_sucia = "   1 ATGCCC GAT TCG   10 ATGCCC   20"
print(f"Original : '{secuencia_sucia}'")

sin_numeros = re.sub(r"\d+", "", secuencia_sucia)
limpia = re.sub(r"\s+", "", sin_numeros).upper()
print(f"Limpia   : '{limpia}'")

In [ ]:
# re.sub: simular transcripción DNA → RNA
dna = "ATGCCCGAATTTAAACGATGA"
rna = re.sub("T", "U", dna)
print(f"DNA : {dna}")
print(f"RNA : {rna}")

---
## 📊 Tabla Resumen de Patrones Biológicos

| Elemento | Patrón regex | Uso |
|----------|-------------|-----|
| Nucleótido cualquiera | `[ATCG]` | Validar secuencias |
| Codón de inicio | `ATG` | Inicio de genes |
| Codones de paro | `TAA\|TAG\|TGA` | Fin de genes |
| ORF (no codicioso) | `ATG[ATCG]+?(TAA\|TAG\|TGA)` | Predicción de genes |
| Microsatélite | `(CA){3,}` | Genética de poblaciones |
| Sitio EcoRI | `GAATTC` | Mapeo de restricción |
| Secuencia válida | `^[ATCG]+$` | Control de calidad |
| N-glicosilación | `N[^P][ST]` | Modificaciones post-traduccionales |
| Señal KDEL | `[KHR]DEL$` | Localización subcelular |
| Dominio TM | `[AVILMFWP]{15,}` | Predicción estructural |

---
*¡Recuerden: el objetivo es entender, no solo ejecutar el código!*